# 基于人类反馈的强化学习 (Reinforcement Learning from Human Feedback --RLHF)

监督微调教会模型遵守指令。但是它不教模型哪个回复更好。

## 问题描述

微调模型生成的两个回复事实上都正确，语法也都看起来没问题。都遵守了输入制定，但是回复A相对于回复B会更好。监督微调没办法识别这种区别，它只训练模型遵守指令生成回复，但是没有机制来判别这个回复比那个更好。它平等地认为每一个回复同样好。此时如果A和B都出现在了微调数据集中，模型会平等的从两个回复中学习。

RLHF 训练了一个奖励模型来预测一个回复是否更收到人类偏好，并且用奖励机制促使模型更倾向于生成高质量的回复，来解决上述问题。

InstructGPT（1.3B参数，但是经过了RLHF）对比GPT-3（175B参数）在85%的场景下，输出的结果更符合人类青睐。

## 基本概念

### 三个阶段

RLHF 不是一个单一的训练过程，而是三个序列状态所组成的流水线，每个都建立在前一个之上。

#### SFT

在一个基础模型上做指令微调。这给你一个能够遵循指令但是不知道哪个回复更好的模型。

#### 奖励模型

收集人类偏好数据：对于同一条提示词生成的两条不同回复，找人类标记“谁更好”。之后训练一个模型来预测这些便好，奖励模型接受提示词、回复作为输入，输出一个打分。

#### PPO （Proximal Policy Optimization 近端策略优化）

使用奖励模型为语言模型生成一个训练信号。语言模型生成回复，奖励模型对回复进行打分，然后PPO更新语言模型以期产生更高的分的回复。使用KL散度惩罚来防止模型更新得离微调后的结果太远，避免把模型训崩。

### 奖励模型

奖励模型的本质是一个语言模型被特化为打分模型。拿到监督微调模型之后，将模型的输出头（就是那个输出词表分布概率的层）替换成一个输出打分表量的头。

训练数据是人类便好的数据对（提示词，被偏好的回复，不被偏好的回复）。

训练损失是 Bradley-Terry 模型
```
loss = -log(sigmoid(reward(perferred) - reward(rejected)))
```

所以上面sigmoid给你了A比B更受偏好的概率，这个损失将奖励模型对受偏好回复的reward打分逐步抬高。

### PPO

PPO 是一种强化学习算法。目标
```
maximize: E[R(prompt, response)] - beta * KL(policy || reference)
```
第一项将模型推向生成打分更高的回复。第二项KL散度则防止模型过远的偏离监督微调结果。

需要KL散度来惩罚的原因是，没有它，模型会找到退化的解。奖励模型是在一个有限的人类偏好数据集上训练的，它有盲点。语言模型会利用那些盲点————找到在奖励模型上得分高，但实际上毫无意义的输出。典型的例子：
* 重复“I'm so helpful and harmless” 在有用性/无害性奖励模型上得分高。
* 产出冗长、听起来正式但是空洞的回复，去模式匹配“高质量”
* 利用那些恰好在训练数据里和高奖励相关的特定短语。

KL惩罚的核心是：你可以改进，但是你不能变成一个完全不同的模型。靠近哪个本来就已经合理的SFT版本，修改得太狠，KL成本就会盖过奖励。

#### PPO 细节

PPO用一个“被裁剪的代理目标”来防止激进的大幅更新。旧策略和新策略之间的比例被裁剪到`[1-eps, 1+eps]`范围内，`eps`的经典取值为0.2。

### 奖励作弊

RLHF的阴暗面。语言模型在针对奖励模型做优化，而奖励模型是人类偏好的不完美代理。随着语言模型越来越擅长最大化奖励，它开始利用奖励模型的弱点。

常见的失败模式：

啰嗦、谄媚、模棱两可、格式作弊

缓解策略：

采用更强的KL惩罚、在对抗样本上训练奖励模型、用多个不同架构的奖励模型

# 开始编码

In [1]:
# preferred: 简短直接回答；rejected: 复述问题 / 冗长绕弯
PREFERENCE_DATA = [
    {
        "prompt": "What is the result of 1+1?",
        "preferred": "2",
        "rejected": "The result of 1+1 is 2.",
    },
    {
        "prompt": "What is the capital of France?",
        "preferred": "Paris",
        "rejected": "The capital of France is Paris.",
    },
    {
        "prompt": "Translate to French: hello",
        "preferred": "bonjour",
        "rejected": "The French translation of hello is bonjour.",
    },
    {
        "prompt": "What does CPU stand for?",
        "preferred": "Central Processing Unit",
        "rejected": "CPU stands for Central Processing Unit.",
    },
    {
        "prompt": "Is 17 a prime number?",
        "preferred": "Yes",
        "rejected": "Yes, 17 is a prime number.",
    },
    {
        "prompt": "Convert 100 Celsius to Fahrenheit.",
        "preferred": "212°F",
        "rejected": "Converting 100 Celsius to Fahrenheit gives 212°F.",
    },
    {
        "prompt": "Name the largest planet in our solar system.",
        "preferred": "Jupiter",
        "rejected": "The largest planet in our solar system is Jupiter.",
    },
    {
        "prompt": "What is 15% of 200?",
        "preferred": "30",
        "rejected": "15% of 200 is equal to 30.",
    },
    {
        "prompt": "Spell 42 in words.",
        "preferred": "forty-two",
        "rejected": "The number 42 spelled in words is forty-two.",
    },
    {
        "prompt": "What year did World War II end?",
        "preferred": "1945",
        "rejected": "World War II ended in the year 1945.",
    },
    {
        "prompt": "Give two synonyms for happy.",
        "preferred": "joyful, cheerful",
        "rejected": "Two synonyms for happy are joyful and cheerful.",
    },
    {
        "prompt": "What is the chemical formula for water?",
        "preferred": "H2O",
        "rejected": "The chemical formula for water is H2O.",
    },
    {
        "prompt": "Write a Python list with numbers 1 to 3.",
        "preferred": "[1, 2, 3]",
        "rejected": "A Python list with numbers 1 to 3 is [1, 2, 3].",
    },
    {
        "prompt": "What does HTTP stand for?",
        "preferred": "HyperText Transfer Protocol",
        "rejected": "HTTP stands for HyperText Transfer Protocol.",
    },
    {
        "prompt": "Who wrote 1984?",
        "preferred": "George Orwell",
        "rejected": "The author who wrote 1984 is George Orwell.",
    },
    {
        "prompt": "What is π to 2 decimal places?",
        "preferred": "3.14",
        "rejected": "π to 2 decimal places is 3.14.",
    },
    {
        "prompt": "Plural of mouse (animal)?",
        "preferred": "mice",
        "rejected": "The plural of mouse (animal) is mice.",
    },
    {
        "prompt": "Translate to Spanish: thank you",
        "preferred": "gracias",
        "rejected": "Translating thank you to Spanish gives gracias.",
    },
    {
        "prompt": "What is 2**10?",
        "preferred": "1024",
        "rejected": "The result of 2**10 is 1024.",
    },
    {
        "prompt": "Name a deep learning framework.",
        "preferred": "PyTorch",
        "rejected": "One deep learning framework you can name is PyTorch.",
    },
    {
        "prompt": "What color do you get by mixing red and blue?",
        "preferred": "Purple",
        "rejected": "Mixing red and blue gives you the color purple.",
    },
    {
        "prompt": "Does the Earth revolve around the Sun?",
        "preferred": "Yes",
        "rejected": "Yes, the Earth does revolve around the Sun.",
    },
    {
        "prompt": "What does NLP stand for?",
        "preferred": "Natural Language Processing",
        "rejected": "NLP stands for Natural Language Processing.",
    },
    {
        "prompt": "Boiling point of water at sea level in C?",
        "preferred": "100°C",
        "rejected": "The boiling point of water at sea level in C is 100°C.",
    },
    {
        "prompt": "Reverse the word hello.",
        "preferred": "olleh",
        "rejected": "Reversing the word hello gives olleh.",
    },
    {
        "prompt": "What is the opposite of ancient?",
        "preferred": "modern",
        "rejected": "The opposite of ancient is modern.",
    },
    {
        "prompt": "SQL: select all rows from users",
        "preferred": "SELECT * FROM users;",
        "rejected": "To select all rows from users, use SELECT * FROM users;",
    },
    {
        "prompt": "What does GPU stand for?",
        "preferred": "Graphics Processing Unit",
        "rejected": "GPU stands for Graphics Processing Unit.",
    },
    {
        "prompt": "Mean of 2, 4, 6?",
        "preferred": "4",
        "rejected": "The mean of 2, 4, 6 is 4.",
    },
    {
        "prompt": "What is 0! ?",
        "preferred": "1",
        "rejected": "The value of 0! is 1.",
    },
]

print(f"preference pairs: {len(PREFERENCE_DATA)}")
print("example:", PREFERENCE_DATA[0])


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RewardModel(nn.Module):
    """打分模型：读完整序列 -> 标量 reward。

    注意：
    - 需要位置编码（否则 Transformer 分不清词序）
    - 不需要因果掩码：RM 不是在生成，而是看完整回答再打分，用双向注意力
    - 用 EncoderLayer，不要 DecoderLayer（Decoder 还要 cross-attn / memory）
    """

    def __init__(
        self,
        vocab_size=256,
        embed_dim=128,
        num_heads=4,
        num_layers=4,
        max_seq_len=128,
        ff_dim=512,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len

        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)

        # 双向自注意力：评分时可同时看到 prompt 与 response 全文
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.score_head = nn.Linear(embed_dim, 1)

    def forward(self, token_ids, attention_mask=None):
        """
        token_ids: (B, T)
        attention_mask: (B, T) 1=valid, 0=pad；可选
        returns: (B,) 每个样本一个标量 reward
        """
        B, T = token_ids.shape
        if T > self.max_seq_len:
            raise ValueError(f"seq_len {T} > max_seq_len {self.max_seq_len}")

        positions = torch.arange(T, device=token_ids.device).unsqueeze(0).expand(B, T)
        h = self.token_embed(token_ids) + self.pos_embed(positions)

        # TransformerEncoder 的 src_key_padding_mask: True 表示忽略（padding）
        pad_mask = None
        if attention_mask is not None:
            pad_mask = attention_mask == 0

        h = self.encoder(h, src_key_padding_mask=pad_mask)
        h = self.norm(h)

        # 取最后一个非 pad token 的隐状态打分（InstructGPT 风格常见做法）
        if attention_mask is None:
            pooled = h[:, -1]
        else:
            lengths = attention_mask.long().sum(dim=1).clamp(min=1)  # (B,)
            last_idx = lengths - 1
            pooled = h[torch.arange(B, device=h.device), last_idx]

        return self.score_head(pooled).squeeze(-1)  # (B,)


def bradley_terry_loss(reward_preferred, reward_rejected):
    """loss = -log σ(r+ - r-)"""
    return -F.logsigmoid(reward_preferred - reward_rejected).mean()


# 快速自检
_rm = RewardModel(vocab_size=100, max_seq_len=32)
_ids = torch.randint(0, 100, (2, 16))
_mask = torch.ones(2, 16)
_mask[0, 12:] = 0
_scores = _rm(_ids, _mask)
assert _scores.shape == (2,), _scores.shape
print("RewardModel ok | scores:", _scores.detach())
